# Load FX Rates from Excel

Reads the FX Rates Excel file and loads it into the `fx_rates` table.

**File Structure:**
- Source file contains two FX rate tables on the same sheet
- **Yellow table** (columns B:D, rows 3-33): Used by this loader
  - Column B: "Currency (To)" - FROM currency (EUR, USD, CAD, etc.)
  - Column C: "Currency (To)" - TO currency (all GBP)
  - Column D: "Rate" - exchange rate
  - ~30 currency pairs

**Note:** The original Excel macro used named range `rng_FX_rates=Rates!$D$4:$D$22` which:
- Only captured column D (rates without currency codes)
- Only captured 19 rows (missing 11 currencies)
- This loader reads the **full range** B3:D33 to capture all data

**Prerequisites:** 
- Upload `FX Rates - Dec 2025.xlsx` to `Files/config/` in your lakehouse
- Source file: `docs/Source Documents/IBM data - HC/IBM data - HC/FX Rates - Dec 2025.xlsx`
- Filename should include period (e.g., "Dec 2025") for automatic period detection

In [ ]:
# ========== Establish Workspace Context ==========
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")

print("[FX RATES] Loading FX rates from Excel file")

In [ ]:
# ========== Imports ==========
from pyspark.sql import functions as F
import pandas as pd
from datetime import datetime

print("[INFO] Imports complete")

## Step 1: Read Excel File

In [ ]:
print("[STEP 1] Reading FX Rates Excel file...")

fx_file_relative = "FX Rates - Dec 2025.xlsx"
fx_file_path = f"Files/config/{fx_file_relative}"

try:
    # Use mssparkutils to find the file
    print(f"  Looking for: {fx_file_path}")
    
    # List config folder to find the file
    try:
        config_files = mssparkutils.fs.ls("Files/config/")
        print(f"  ✓ Found {len(config_files)} file(s) in Files/config/")
        
        # Find the FX rates file
        fx_file_full_path = None
        for f in config_files:
            print(f"    - {f.name}")
            if f.name.lower() == fx_file_relative.lower():
                fx_file_full_path = f.path
                print(f"  ✓ Located: {f.name}")
        
        if fx_file_full_path is None:
            raise FileNotFoundError(f"'{fx_file_relative}' not found in Files/config/")
        
        # Read the FX rates table from the yellow table (columns B:D, starting row 3)
        # Header row is row 3 (0-indexed row 2)
        # Data rows: 4-33 (or until blank rows)
        print(f"\n  Reading FX rates from yellow table (columns B:D)...")
        
        # Read specific columns B:D (usecols="B:D"), header at row 2 (0-indexed)
        pdf_fx = pd.read_excel(fx_file_full_path, usecols="B:D", header=2)
        
        print(f"  Raw data loaded: {len(pdf_fx)} rows, {len(pdf_fx.columns)} columns")
        print(f"  Column names: {pdf_fx.columns.tolist()}")
        
        # Clean column names
        pdf_fx.columns = [str(col).strip() for col in pdf_fx.columns]
        
        # Remove rows where all values are null (empty rows at bottom)
        pdf_fx = pdf_fx.dropna(how='all')
        
        # Remove rows where rate is null (incomplete data)
        pdf_fx = pdf_fx.dropna(subset=[pdf_fx.columns[2]])  # Column D (Rate)
        
        print(f"\n✓ File loaded: {len(pdf_fx)} rows after cleaning")
        print(f"\nColumns: {pdf_fx.columns.tolist()}")
        print(f"\nFirst 10 rows:")
        print(pdf_fx.head(10))
        print(f"\nLast 5 rows:")
        print(pdf_fx.tail(5))
        
    except Exception as e:
        print(f"  ✗ Error accessing Files/config/: {str(e)}")
        raise
    
except FileNotFoundError as e:
    print(f"✗ File not found: {str(e)}")
    print("\n[ACTION REQUIRED]:")
    print("1. In your Fabric lakehouse, go to Files section")
    print("2. Verify you're in the 'config' folder (shown in your screenshot)")
    print("3. Verify 'FX Rates - Dec 2025.xlsx' is visible there")
    print("4. If not, upload from: docs/Source Documents/IBM data - HC/IBM data - HC/")
    raise
except Exception as e:
    print(f"✗ Error reading file: {str(e)}")
    raise

## Step 2: Validate Column Structure

The yellow FX rates table should have 3 columns:
1. **Currency (To)** - actually the FROM currency (EUR, USD, CAD, etc.)
2. **Currency (To)** - the TO currency (all GBP)
3. **Rate** - the exchange rate

In [ ]:
print("[STEP 2] Validating column structure...")

print(f"\nColumns found: {pdf_fx.columns.tolist()}")
print(f"Number of columns: {len(pdf_fx.columns)}")

if len(pdf_fx.columns) != 3:
    print(f"\n⚠️  WARNING: Expected 3 columns, found {len(pdf_fx.columns)}")
    print(f"   Continuing with available columns...")

# Show sample data
print(f"\nSample data (first 5 rows):")
print(pdf_fx.head(5))

print(f"\n✓ Ready for transformation")

## Step 3: Transform to Target Schema

Target schema: `period, from_currency, to_currency, rate, effective_date, source, updated_at`

Assuming the file naming convention includes the period (e.g., "FX Rates - Dec 2025.xlsx")

In [ ]:
print("[STEP 3] Transforming to target schema...")

# Create a working copy
pdf_final = pdf_fx.copy()

# Rename columns to standard names
# Column 0: from_currency (e.g., EUR, USD, CAD)
# Column 1: to_currency (should all be GBP)
# Column 2: rate
if len(pdf_final.columns) >= 3:
    pdf_final.columns = ['from_currency', 'to_currency', 'rate']
    print(f"  ✓ Renamed columns to standard names")
else:
    print(f"  ✗ ERROR: Expected 3 columns, found {len(pdf_final.columns)}")
    raise ValueError(f"Invalid column count: {len(pdf_final.columns)}")

# Clean up data types
pdf_final['from_currency'] = pdf_final['from_currency'].astype(str).str.strip().str.upper()
pdf_final['to_currency'] = pdf_final['to_currency'].astype(str).str.strip().str.upper()
pdf_final['rate'] = pd.to_numeric(pdf_final['rate'], errors='coerce')

print(f"  ✓ Cleaned data types")

# Extract period from filename (e.g., "Dec 2025" -> "2025-12")
# Parse "FX Rates - Dec 2025.xlsx" -> period "2025-12"
import re
month_abbr_to_num = {
    'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04',
    'may': '05', 'jun': '06', 'jul': '07', 'aug': '08',
    'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
}

period_str = None
effective_date_str = None

# Try to extract month and year from filename
match = re.search(r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+(\d{4})', fx_file_relative, re.IGNORECASE)
if match:
    month_abbr = match.group(1).lower()
    year = match.group(2)
    month_num = month_abbr_to_num.get(month_abbr, '12')
    period_str = f"{year}-{month_num}"
    # Use last day of month as effective date
    effective_date_str = f"{year}-{month_num}-{pd.Period(period_str).days_in_month}"
    print(f"  ✓ Extracted period from filename: {period_str}")
    print(f"  ✓ Effective date: {effective_date_str}")
else:
    # Default to current month if can't parse
    from datetime import datetime
    period_str = datetime.now().strftime('%Y-%m')
    effective_date_str = datetime.now().strftime('%Y-%m-%d')
    print(f"  ⚠️  Could not extract period from filename, using current: {period_str}")

# Add derived columns
pdf_final['period'] = period_str
pdf_final['effective_date'] = effective_date_str
pdf_final['source'] = fx_file_relative

print(f"  ✓ Added metadata columns")

# Remove rows with null rates or invalid currencies
initial_count = len(pdf_final)
pdf_final = pdf_final.dropna(subset=['rate', 'from_currency', 'to_currency'])
pdf_final = pdf_final[pdf_final['rate'] > 0]  # Remove zero or negative rates
removed_count = initial_count - len(pdf_final)

if removed_count > 0:
    print(f"  ⚠️  Removed {removed_count} invalid rows")

# Reorder columns to match target schema
pdf_final = pdf_final[['period', 'from_currency', 'to_currency', 'rate', 'effective_date', 'source']]

print(f"\n  ✓ Transformed to {len(pdf_final)} rows")
print(f"  \nSample of transformed data:")
print(pdf_final.head(10))
print(f"\nLast 5 rows:")
print(pdf_final.tail(5))

print(f"\n  Final row count: {len(pdf_final)}")
print(f"  Unique from_currencies: {pdf_final['from_currency'].nunique()} ({', '.join(sorted(pdf_final['from_currency'].unique()[:10]))}{'...' if pdf_final['from_currency'].nunique() > 10 else ''})")
print(f"  Unique to_currencies: {pdf_final['to_currency'].nunique()} ({', '.join(sorted(pdf_final['to_currency'].unique()))})")
print(f"  Rate range: {pdf_final['rate'].min():.6f} to {pdf_final['rate'].max():.6f}")

## Step 4: Convert to Spark DataFrame

In [ ]:
print("[STEP 4] Converting to Spark DataFrame...")

# Convert to Spark
df_fx = spark.createDataFrame(pdf_final)

# Add timestamp
df_fx = df_fx.withColumn("updated_at", F.current_timestamp())

# Ensure rate is DOUBLE type
df_fx = df_fx.withColumn("rate", F.col("rate").cast("double"))

print(f"✓ Spark DataFrame created: {df_fx.count()} rows")
print(f"\nSchema:")
df_fx.printSchema()

print(f"\nSample data (first 10 rows):")
df_fx.show(10, truncate=False)

print(f"\nLast 5 rows:")
df_fx.orderBy(F.col("from_currency").desc()).show(5, truncate=False)

## Step 5: Load into fx_rates Table

In [ ]:
print("[STEP 5] Loading into fx_rates table...")

# Check if table exists
try:
    existing_count = spark.table("fx_rates").count()
    print(f"  Existing rows in fx_rates: {existing_count}")
except:
    print("  ✗ fx_rates table does not exist")
    print("  [ACTION] Run nb_setup_tables.ipynb first to create the table")
    raise

# Load data
df_fx.write.mode("append").saveAsTable("fx_rates")

new_count = spark.table("fx_rates").count()
loaded_count = new_count - existing_count

print(f"\n✓ Loaded {loaded_count} new FX rates")
print(f"  Total rows in fx_rates: {new_count}")

## Step 6: Verify Loaded Data

In [ ]:
print("[STEP 6] Verifying loaded data...")

df_verify = spark.sql("""
    SELECT 
        period,
        COUNT(*) as rate_count,
        COUNT(DISTINCT from_currency) as from_currencies,
        COUNT(DISTINCT to_currency) as to_currencies,
        MIN(rate) as min_rate,
        MAX(rate) as max_rate
    FROM fx_rates
    GROUP BY period
    ORDER BY period DESC
""")

print("\nFX Rates Summary:")
df_verify.show(truncate=False)

print("\n[SUCCESS] FX rates loaded successfully")